In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

<font size="5" color="black">ch7. OpenAI Responses API</font>

# OpenAI Responses API를 활용한 서비스 구현


# 1. OpenAI Responses API 소개

* Responses API : 역할 및 질문 전달을 통한 즉각적인 답변 수신 구조

* 주요 기능 및 특징
    * 대화 지속성 : 직전 응답의 `response.id`를 `previous_response_id`로 전달하여 별도 저장 공간 없이 자동 유지되는 대화 맥락
    * 다양한 도구 통합 : 코드 실행(Code Interpreter), 문서 검색(File Search), 웹 검색(Web Search), 함수 호출(Function Calling) 연결을 통한 데이터 계산·파일 처리·외부 검색 작업의 자동화
    * 개인화된 지시어 : `instructions` 파라미터를 활용한 모델의 성격, 역할, 톤 및 도메인 지식 설정
    * 간결한 응답 접근 : `response.output_text`를 통한 최종 텍스트 답변의 즉각적인 추출

* 핵심 가치 및 활용성
    * 대화 상태 관리, 외부 도구 통합, 문서 검색의 플랫폼 위임을 통한 개발자의 핵심 로직 구현 집중
    * 챗봇 및 자동화 에이전트 구축을 위한 강력한 개발 솔루션 제공

> Note: API는 계속 발전하므로, 사용 전 [공식 문서](https://platform.openai.com/docs/api-reference/responses)를 확인하는 습관을 들이는 게 좋습니다.

# 2. API 키 설정 및 환경 변수 사용

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

# 3. 기본적인 응답 생성 및 메시지 보내기

- `client.responses.create()` : 한 번의 호출로 역할(instructions)과 질문(input)을 동시에 넘겨 바로 답변을 받습니다.

In [ ]:
# 1. 첫 번째 응답 생성 (초기 지침 및 질문)
response = client.responses.create(
    model='gpt-4.1-nano',
    instructions='당신은 유능하고 친절한 도움말 어시스턴트입니다. 질문에 20자 내로 친절하게 답변하세요',
    input='오늘 서울 날씨는 몇 도까지 올라가나요?',
    # store=True : (기본값) 서버에 대화 내역 저장 ; False 설정 시 previous_response_id 사용 불가
)

# 응답 ID 확인 : 대화를 이어가기 위해 previous_response_id로 사용 필요
print("response's id : ", response.id[:5])

# 최종 답변 텍스트
print("Assistant : ", response.output_text)

In [ ]:
response.usage.total_tokens

```
위 코드에서는 `client.responses.create` 메서드를 사용해 응답을 생성했습니다. 주요 파라미터를 살펴보면:
```

- model : 응답 생성에 사용할 언어 모델을 지정

    GPT-4 계열
    - gpt-4o - 최신 멀티모달 모델
    - gpt-4o-mini - 경량화된 GPT-4o (비용 효율적, 이 튜토리얼의 기본값)
    - gpt-4-turbo - 더 빠르고 효율적인 GPT-4
    - gpt-4 - 기본 GPT-4 모델

    GPT-3.5 계열
    - gpt-3.5-turbo - 빠르고 효율적인 모델

* `instructions` (시스템 레벨 지시어)
    * 모델의 기본 역할 및 반응 규칙 정의
    * 매 API 호출 시 지속적인 전달 필요성
    * 자주 사용하는 역할에 대한 변수 또는 함수화 재사용 권장


* `input` (사용자 입력) :  사용자의 질문 및 대화 내용 전달


* `tools` (활성화 도구 목록)
    * 모델에서 사용 가능한 도구 목록 지정
    * 기본 Q&A 실행 시 생략 가능
    * 코드 인터프리터 및 파일 검색 기능 사용 시 설정 필요

```
응답이 생성되면 고유 ID(`response.id`)가 반환됩니다. 이 ID는 이후 대화를 이어갈 때 `previous_response_id`로 지정하면 되므로 저장해둡니다.
```

> 참고 : 각 응답(response)은 `previous_response_id`로 바로 이전 응답을 가리키는 **체인(사슬) 구조**를 이룹니다. OpenAI 서버가 이 체인을 따라가며 이전 대화 내용을 자동으로 반영해주기 때문에, 개발자가 대화 내역을 매번 직접 다시 보낼 필요가 없습니다.

## 대화 이어가기 (멀티턴)

* `previous_response_id` (이전 응답 식별자)
    * 직전 답변의 `response.id` 값 전달
    * 이전 대화 맥락 유지 및 연속적인 답변 생성 기능

In [ ]:
# 2. 대화 이어가기 (이전 맥락 및 instructions 상속)
follow_up = client.responses.create(
    model='gpt-4.1-nano',
    input='진짜 정확한걸까?',
    previous_response_id=response.id # 이전 응답 ID를 참조하여 서버 측 대화 히스토리 연결
)

print('Assistant :', follow_up.output_text)

### 예제: 대화형 챗봇 (interactive chatbot)

사용자가 계속 입력하고, 종료 명령을 입력할 때까지 대화를 이어가는 챗봇을 구현해보겠습니다.

역할(instructions)을 고정해두고, 매 턴마다 `previous_response_id`로 직전 응답을 이어붙이며 `responses.create()`를 반복 호출하는 구조입니다. 대화가 끝나면 이력을 파일로 저장합니다.



In [ ]:
# 대화 이력을 저장할 수 있는 챗봇 생성

from openai import OpenAI, OpenAIError
import time

# 1. 고객 지원 챗봇의 instruction 정의

cs_instructions = '당신은 고객 지원 챗봇 입니다. 사용자 문의에 친절하고 간략히, 정확하게 답하세요'

print('챗봇 시작 (exit 입력 시 종료)')
conv_log = [] # 대화 종료 시 대화 로그 저장 용도 ; 형태 : [(role, text, time)]
previous_id = None # 대화 시작 시 이전 response의 id는 None

while True:
    user_input = input('User : ').strip()
    if user_input.lower() in ('exit',):
        print('Assistant : 이용해주셔서 감사합니다.')
        break
    if user_input.strip() == '':
        continue
        
    now = time.time()
    conv_log.append(('user', user_input, now))
    
# 2. 응답 생성 요청 (이전 대화가 있을 경우 이어가기)

    try:
        response = client.responses.create(
            model='gpt-4.1-nano',
            instructions=cs_instructions,
            input=user_input,
            previous_response_id=previous_id
        )
    except OpenAIError as e:
        print('오류가 발생했습니다.', e)
        continue
        
# 3. 다음 호출을 위해 이번 대화의 id 저장 후 답변 출력 및 로그 저장
    previous_id = response.id
    reply_text = response.output_text
    print('Assistant :', reply_text)
    conv_log.append(('assistant', reply_text, time.time()))
    
    # previous_id 사용 시 이전 log가 입력 토큰에 포함됨
    print('입력 토큰 :', response.usage.input_tokens)
    print('출력 토큰 :', response.usage.output_tokens)

In [ ]:
# 4. 대화 로그 백업
file_name = 'data/ch7_chat_log.txt'
print(f'대화 로그를 저장합니다. 파일명 : ({file_name})')

with open(file_name, 'w', encoding='utf-8') as f:
    for role, text, create_at in conv_log:
        dateStyle_str = time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(create_at))
        row = f'{role:9}({dateStyle_str}) : {text}\n'
        f.write(row)
        print(row, end='')

# 4. 파일 업로드 및 코드 인터프리터 활용

* 파일 업로드 기능의 개념 및 활용
    * 사용자가 제공한 데이터를 모델이 분석·활용할 수 있도록 지원하는 핵심 기능
    * CSV 데이터 분석, 텍스트 파일 내부 숫자 합산 등의 다양한 작업 수행 가능


**단계별 처리 방식 및 절차**
1. `client.files.create()` 호출을 통한 파일 업로드 및 `file_id` 획득
2. `tools=[{"type": "code_interpreter", "container": {"type": "auto", "file_ids": [file_id]}}]` 구문을 활용한 코드 인터프리터 도구 직접 연동
3. 코드 인터프리터 실행 시 결과 산출 및 `container_id` 반환
4. 반환된 컨테이너(가상 실행 환경) 내부에서의 실제 Python 코드 실행

> Tips: 컨테이너는 일정 시간(기본 약 1시간, 유휴 20분)이 지나면 만료됩니다. 만료된 컨테이너는 복구할 수 없으므로, 새 요청에서 파일이 다시 필요하면 `container.file_ids`에 같은 `file_id`를 다시 넣어주면 됩니다. (파일 자체를 재업로드할 필요는 없습니다.)


In [ ]:
# 1. 파일 업로드 및 파일 id 생성
file_obj = client.files.create(
    file = open('data/data.csv', 'rb'),
    purpose='assistants'
)

print('uploaded file ID :', file_obj.id[:4])

# 2. 업로드한 파일을 코드 인터프리터 도구에 연결하여 응답 요청
question = '첨부된 파일을 읽고, 수치 데이터의 요약본을 출력해줘'

response = client.responses.create(
    model='gpt-4o-mini',
    instructions='당신은 데이터 분석을 돕는 어시스턴트 입니다. 필요 시 python 코드를 작성하고 결과를 요약해주세요.',
    input=question,
    tools=[{
        'type':'code_interpreter',
        'container':{'type':'auto','file_ids':[file_obj.id]} # 코드 인터프리터의 파일 접근 허가
    }]
)

# 3. 최종 답변
print('Assistant :', response.output_text)

# 5. 파일 검색 (File Search) 활용
- 발전한 형태 : RAG 적용

* File Search 개념 및 핵심 역할
  - 다수의 문서를 대상으로 관련 정보를 탐색하여 답변을 생성하는 도구
  - Vector Store(검색 가능한 형태로 문서를 저장하는 공간) 생성 후 파일 추가 및 ID 지정 방식 활용

* 단계별 수행 절차 및 실행 방식
  1. client.vector_stores.create() 호출을 통한 Vector Store 생성
  2. client.vector_stores.files.upload_and_poll() 활용 파일 업로드 및 임베딩·인덱싱 완료 대기
  3. tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}] 설정을 통한 도구 지정 및 질문 수행

In [ ]:
# 1. vector store 생성 (문서를 검색 가능한 형태로 변환 및 저장하는 역할)
vector_store = client.vector_stores.create(name='quiz_store')
print("vector store's id", vector_store.id[:4])

# 2. vector store에 파일 업로드
with open('data/ch06_quiz.txt', 'rb') as f:
    client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store.id,
        file=f
    )

In [ ]:
# 3. file_search 도구를 활용 (vector_id 지정)
search_response = client.responses.create(
    model='gpt-4o-mini',
    instructions='첨부된 문서를 근거로 답하세요',
    input='커피는 무슨 맛이야', # 문서에 없는 내용의 경우, '정보를 찾을 수 없음'
    tools=[{
        'type':'file_search',
        'vector_store_ids':[vector_store.id]
    }]
)
print(search_response.output_text)